# Convert EEG XDF To FIF
Use `mne` with `pyxdf` to convert EEG streams from `.xdf` files into `.fif` files saved next to the source files.

In [8]:
from pathlib import Path

import mne
import numpy as np
import pandas as pd
import pyxdf

In [2]:
data_root = Path(r"./0-back/eeg")
xdf_files = sorted(data_root.rglob("*_eeg.xdf"))

print(f"Found {len(xdf_files)} XDF file(s)")
for path in xdf_files[:5]:
    print(path)

Found 4 XDF file(s)
0-back\eeg\sub-P001\ses-S001\eeg\sub-P001_ses-S001_task-Default_run-001_eeg.xdf
0-back\eeg\sub-P002\ses-S001\eeg\sub-P002_ses-S001_task-Default_run-001_eeg.xdf
0-back\eeg\sub-P003\ses-S001\eeg\sub-P003_ses-S001_task-Default_run-001_eeg.xdf
0-back\eeg\sub-P004\ses-S001\eeg\sub-P004_ses-S001_task-Default_run-001_eeg.xdf


In [3]:
def _channel_list_from_stream(stream):
    info = stream.get("info", {})
    desc = info.get("desc", [])
    if not desc:
        return []

    channels = desc[0].get("channels", [])
    if not channels:
        return []

    channel_entries = channels[0].get("channel", [])
    if isinstance(channel_entries, dict):
        channel_entries = [channel_entries]

    return channel_entries


def _channel_name(channel, index):
    for key in ("label", "name"):
        value = channel.get(key, [])
        if value:
            return str(value[0])
    return f"EEG{index + 1:03d}"


def _channel_type(channel):
    value = channel.get("type", [])
    if not value:
        return "eeg"

    channel_type = str(value[0]).lower()
    if channel_type in {"eeg", "eog", "ecg", "emg", "misc", "stim"}:
        return channel_type
    return "eeg"


def pick_eeg_stream(streams):
    ranked = []
    for stream in streams:
        info = stream.get("info", {})
        name = str(info.get("name", [""])[0]).lower()
        stream_type = str(info.get("type", [""])[0]).lower()
        channel_count = int(float(info.get("channel_count", [0])[0]))
        nominal_srate = float(info.get("nominal_srate", [0])[0])

        score = 0
        if stream_type == "eeg":
            score += 3
        if "eeg" in name:
            score += 2
        if channel_count > 4:
            score += 1
        if nominal_srate > 0:
            score += 1

        ranked.append((score, channel_count, nominal_srate, stream))

    ranked.sort(key=lambda item: (item[0], item[1], item[2]), reverse=True)
    if not ranked or ranked[0][0] == 0:
        raise ValueError("Could not identify an EEG stream in this XDF file.")
    return ranked[0][3]


def xdf_eeg_to_raw(xdf_path):
    streams, _ = pyxdf.load_xdf(str(xdf_path))
    stream = pick_eeg_stream(streams)

    info = stream["info"]
    sfreq = float(info["nominal_srate"][0])
    data = np.asarray(stream["time_series"], dtype=float)

    if data.ndim != 2:
        raise ValueError(f"Unexpected EEG data shape: {data.shape}")

    channel_count = int(float(info["channel_count"][0]))
    if data.shape[1] == channel_count:
        data = data.T
    elif data.shape[0] != channel_count:
        raise ValueError(f"Could not align channels for data shape {data.shape} and channel count {channel_count}")

    channel_entries = _channel_list_from_stream(stream)
    ch_names = [_channel_name(channel, idx) for idx, channel in enumerate(channel_entries)]
    ch_types = [_channel_type(channel) for channel in channel_entries]

    if len(ch_names) != data.shape[0]:
        ch_names = [f"EEG{idx + 1:03d}" for idx in range(data.shape[0])]
        ch_types = ["eeg"] * data.shape[0]

    mne_info = mne.create_info(ch_names=ch_names, sfreq=sfreq, ch_types=ch_types)
    raw = mne.io.RawArray(data, mne_info)
    raw.set_meas_date(None)
    return raw, stream


def convert_xdf_file(xdf_path, overwrite=False):
    xdf_path = Path(xdf_path)
    fif_path = xdf_path.with_name(f"{xdf_path.stem}_raw.fif")

    raw, stream = xdf_eeg_to_raw(xdf_path)
    raw.save(fif_path, overwrite=overwrite)

    stream_info = stream["info"]
    return {
        "xdf": str(xdf_path),
        "fif": str(fif_path),
        "stream_name": stream_info.get("name", [""])[0],
        "stream_type": stream_info.get("type", [""])[0],
        "n_channels": raw.info["nchan"],
        "sfreq": raw.info["sfreq"],
        "duration_sec": raw.times[-1] if raw.n_times else 0,
    }

In [4]:
sample_result = convert_xdf_file(xdf_files[0], overwrite=True)
sample_result

Creating RawArray with float64 data, n_channels=20, n_times=613243
    Range : 0 ... 613242 =      0.000 ...  1226.484 secs
Ready.
Writing c:\Users\n.saleem\Documents\GitHub\0x2x2dual_WCST_analysis\data\0-back\eeg\sub-P001\ses-S001\eeg\sub-P001_ses-S001_task-Default_run-001_eeg_raw.fif
Closing c:\Users\n.saleem\Documents\GitHub\0x2x2dual_WCST_analysis\data\0-back\eeg\sub-P001\ses-S001\eeg\sub-P001_ses-S001_task-Default_run-001_eeg_raw.fif
[done]


{'xdf': '0-back\\eeg\\sub-P001\\ses-S001\\eeg\\sub-P001_ses-S001_task-Default_run-001_eeg.xdf',
 'fif': '0-back\\eeg\\sub-P001\\ses-S001\\eeg\\sub-P001_ses-S001_task-Default_run-001_eeg_raw.fif',
 'stream_name': 'LSLOutletStreamName-EEG',
 'stream_type': 'EEG',
 'n_channels': 20,
 'sfreq': 500.0,
 'duration_sec': np.float64(1226.484)}

In [7]:
results = []
for xdf_path in xdf_files:
    try:
        results.append(convert_xdf_file(xdf_path, overwrite=True))
    except Exception as exc:
        results.append({"xdf": str(xdf_path), "error": str(exc)})

pd.DataFrame(results) if results else "No files found"

Creating RawArray with float64 data, n_channels=20, n_times=613243
    Range : 0 ... 613242 =      0.000 ...  1226.484 secs
Ready.
Overwriting existing file.
Writing c:\Users\n.saleem\Documents\GitHub\0x2x2dual_WCST_analysis\data\0-back\eeg\sub-P001\ses-S001\eeg\sub-P001_ses-S001_task-Default_run-001_eeg_raw.fif
Overwriting existing file.
Closing c:\Users\n.saleem\Documents\GitHub\0x2x2dual_WCST_analysis\data\0-back\eeg\sub-P001\ses-S001\eeg\sub-P001_ses-S001_task-Default_run-001_eeg_raw.fif
[done]
Creating RawArray with float64 data, n_channels=20, n_times=620743
    Range : 0 ... 620742 =      0.000 ...  1241.484 secs
Ready.
Overwriting existing file.
Writing c:\Users\n.saleem\Documents\GitHub\0x2x2dual_WCST_analysis\data\0-back\eeg\sub-P002\ses-S001\eeg\sub-P002_ses-S001_task-Default_run-001_eeg_raw.fif
Overwriting existing file.
Closing c:\Users\n.saleem\Documents\GitHub\0x2x2dual_WCST_analysis\data\0-back\eeg\sub-P002\ses-S001\eeg\sub-P002_ses-S001_task-Default_run-001_eeg_raw.fif

,xdf,fif,stream_name,stream_type,n_channels,sfreq,duration_sec
0,0-back\eeg\sub-P001\ses-S001\eeg\sub-P001_ses-...,0-back\eeg\sub-P001\ses-S001\eeg\sub-P001_ses-...,LSLOutletStreamName-EEG,EEG,20,500.0,1226.484
1,0-back\eeg\sub-P002\ses-S001\eeg\sub-P002_ses-...,0-back\eeg\sub-P002\ses-S001\eeg\sub-P002_ses-...,LSLOutletStreamName-EEG,EEG,20,500.0,1241.484
2,0-back\eeg\sub-P003\ses-S001\eeg\sub-P003_ses-...,0-back\eeg\sub-P003\ses-S001\eeg\sub-P003_ses-...,LSLOutletStreamName-EEG,EEG,20,500.0,1205.976
3,0-back\eeg\sub-P004\ses-S001\eeg\sub-P004_ses-...,0-back\eeg\sub-P004\ses-S001\eeg\sub-P004_ses-...,LSLOutletStreamName-EEG,EEG,20,500.0,1191.964
